# 05 — State C Post-FEA Revision

Use the real FreeCAD FEM results to generate the post-FEA CAD revision and compare it to State B.


In [ ]:
from base64 import b64encode
from pathlib import Path
import json
import logging
import os
import sys

import yaml
from IPython.display import HTML, Image, Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "code_base").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
MODULE_ROOT = PROJECT_ROOT / "code_base" / "fea_cad_one_sample"
if str(MODULE_ROOT) not in sys.path:
    sys.path.insert(0, str(MODULE_ROOT))

API_ENV_PATH = MODULE_ROOT / "src" / "api.env"


def _load_api_env(path: Path) -> dict[str, str]:
    """Load a local api.env file into the current process environment."""

    env_values: dict[str, str] = {}
    if not path.exists():
        return env_values
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip("'").strip('"')
        if key and value:
            env_values[key] = value
            os.environ.setdefault(key, value)
    return env_values


api_env_values = _load_api_env(API_ENV_PATH)

from src import interfaces as api

logging.basicConfig(level=logging.INFO, format="%(name)s | %(levelname)s | %(message)s")

def _img(path: Path, width: int = 260) -> str:
    data = b64encode(Path(path).read_bytes()).decode("ascii")
    return f'<img src="data:image/png;base64,{data}" width="{width}" />'

def display_image_row(title: str, items: list[tuple[str, Path]], width: int = 260) -> None:
    cells = []
    for label, path in items:
        cells.append(f'<td style="padding:8px;text-align:center;vertical-align:top"><div><b>{label}</b></div>{_img(path, width)}</td>')
    html = f'<h3>{title}</h3><table><tr>{"".join(cells)}</tr></table>'
    display(HTML(html))

FIXED_SAMPLE_PATH = MODULE_ROOT / "experiment_config" / "fixed_sample.yaml"
OUTPUT_ROOT = MODULE_ROOT / "outputs"
sample_config = yaml.safe_load(FIXED_SAMPLE_PATH.read_text(encoding="utf-8"))
sample_id = str(sample_config["sample_id"])
connection_string = os.environ["CAD_DB_CONNECTION_STRING"]
state_a_dir = OUTPUT_ROOT / f"sample_{sample_id}" / "01_dataset_original"
state_b_dir = OUTPUT_ROOT / f"sample_{sample_id}" / "02_fea_constrained_revision"
state_c_dir = OUTPUT_ROOT / f"sample_{sample_id}" / "05_post_fea_revision"
comparison_dir = OUTPUT_ROOT / f"sample_{sample_id}" / "03_comparison"
manual_dir = OUTPUT_ROOT / f"sample_{sample_id}" / "04_manual_freecad_fea"
screenshots_dir = manual_dir / "screenshots"
state_c_dir.mkdir(parents=True, exist_ok=True)
comparison_dir.mkdir(parents=True, exist_ok=True)

pipeline_config = api.PipelineConfig(
    config_name="config_gpt_5_4_mini.yaml",
    config_path=MODULE_ROOT / "src" / "copied_from_cadcodeverify" / "configs" / "config_gpt_5_4_mini.yaml",
    output_root=OUTPUT_ROOT,
    force=True,
)

print("[STAGE] Setup")
print(f"  → sample id : {sample_id}")
print(f"  → state B   : {state_b_dir}")
print(f"  → state C   : {state_c_dir}")
print(f"  → manual dir: {manual_dir}")
print(f"  → api.env   : {API_ENV_PATH}")
print(f"  → api.env keys: {sorted(api_env_values)}")


In [ ]:
print("[STAGE] Load State B and manual FEA evidence")
state_b_code_path = state_b_dir / "fea_revision_code.py"
state_b_step_path = state_b_dir / "fea_revision.step"
state_b_stl_path = state_b_dir / "fea_revision.stl"
state_b_views_dir = state_b_dir / "views"
state_b_change_log_path = state_b_dir / "fea_revision_change_log.json"
state_b_prompt_path = state_b_dir / "fea_revision_prompt.txt"
load_case_path = state_b_dir / "load_case.json"
manual_report_path = manual_dir / "fea_report.json"
required_screenshots = [screenshots_dir / name for name in ["mesh.png", "fixed_region.png", "load_region.png", "von_mises.png", "displacement.png"]]

assert state_b_code_path.exists()
assert state_b_step_path.exists()
assert state_b_stl_path.exists()
assert load_case_path.exists()
assert manual_report_path.exists()

load_case = api.LoadCase(**json.loads(load_case_path.read_text(encoding="utf-8")))
fea_revision_code = state_b_code_path.read_text(encoding="utf-8")
fea_report = json.loads(manual_report_path.read_text(encoding="utf-8"))
validation = api.validate_manual_fea_completion(fea_report, required_screenshots)

print(f"  → complete        : {validation['is_complete']}")
print(f"  → missing fields  : {validation['missing_fields']}")
print(f"  → missing evidence: {validation['missing_evidence_paths']}")
display_image_row("State B views", [(name, state_b_views_dir / f"{name}.png") for name in ["front", "side", "top", "iso", "grid"]])
for path in required_screenshots:
    if path.exists():
        display(Markdown(f"**{path.name}**"))
        display(Image(filename=str(path)))

state_c_ready = validation["is_complete"]
if not state_c_ready:
    print("⛔ State C blocked: finish manual FEA evidence before revising the CAD again.")


In [ ]:
print("[STAGE] Generate State C revision when gate passes")
post_fea_result = None
post_fea_prompt_path = state_c_dir / "post_fea_prompt.txt"
post_fea_code_path = state_c_dir / "post_fea_code.py"
post_fea_change_log_path = state_c_dir / "post_fea_change_log.json"
post_fea_provenance_path = state_c_dir / "provenance.json"
post_fea_step_path = state_c_dir / "post_fea.step"
post_fea_stl_path = state_c_dir / "post_fea.stl"
post_fea_views_dir = state_c_dir / "views"
post_fea_side_by_side_path = state_c_dir / "state_b_vs_state_c.png"
metrics_json_path = state_c_dir / "geometry_metrics.json"
metrics_md_path = state_c_dir / "geometry_metrics.md"

if state_c_ready:
    if not post_fea_code_path.exists() or not post_fea_change_log_path.exists() or not post_fea_provenance_path.exists():
        if not os.environ.get("OPENAI_API_KEY"):
            raise RuntimeError(f"OPENAI_API_KEY is required to generate State C when artifacts do not already exist. Populate {API_ENV_PATH} or export the variable before running this cell.")
        post_fea_result = api.revise_code_after_fea(fea_revision_code, load_case, fea_report, required_screenshots, pipeline_config)
    if post_fea_result is not None:
        post_fea_prompt_path = post_fea_result.prompt_path
        post_fea_code_path = post_fea_result.code_path
        post_fea_change_log_path = post_fea_result.change_log_path
        post_fea_provenance_path = post_fea_result.provenance_path
        post_fea_step_path = post_fea_result.step_path
        post_fea_stl_path = post_fea_result.stl_path
    post_fea_code = post_fea_code_path.read_text(encoding="utf-8")
    if not post_fea_step_path.exists() or not post_fea_stl_path.exists():
        execution = api.execute_and_export_post_fea_revision_cadquery(post_fea_code, state_c_dir, force=True)
        post_fea_step_path = Path(execution["step_path"])
        post_fea_stl_path = Path(execution["stl_path"])
    view_paths = api.render_views(post_fea_stl_path, post_fea_views_dir, force=True)
    api.build_side_by_side_comparison(state_b_views_dir, post_fea_views_dir, post_fea_side_by_side_path, force=True)
    geometry_metrics = api.compute_geometry_metrics({"State B": state_b_stl_path, "State C": post_fea_stl_path}, metrics_json_path, force=True)
    api.build_geometry_metrics_markdown(geometry_metrics, metrics_md_path, force=True)
    display(Markdown("## State B vs State C"))
    display(Image(filename=str(post_fea_side_by_side_path)))
    display(Markdown("## Post-FEA prompt"))
    display(Markdown(f"```text\n{post_fea_prompt_path.read_text(encoding='utf-8')}\n```"))
    display(Markdown("## State C geometry metrics"))
    display(Markdown(metrics_md_path.read_text(encoding="utf-8")))
    post_fea_change_log = json.loads(post_fea_change_log_path.read_text(encoding="utf-8"))
    print(f"  → State C code   : {post_fea_code_path}")
    print(f"  → State C step   : {post_fea_step_path}")
    print(f"  → State C stl    : {post_fea_stl_path}")
    print(f"  → State C change : {post_fea_change_log_path}")
    print(f"  → State C views  : {sorted(view_paths.keys())}")
else:
    post_fea_change_log = {}
    print("State C remains blocked until manual FEA evidence is complete.")


In [ ]:
print("[STAGE] Assertions")
if state_c_ready:
    assert post_fea_prompt_path.exists()
    assert post_fea_code_path.exists()
    assert post_fea_change_log_path.exists()
    assert post_fea_provenance_path.exists()
    assert post_fea_step_path.exists()
    assert post_fea_stl_path.exists()
    assert post_fea_side_by_side_path.exists()
    assert metrics_json_path.exists()
    assert metrics_md_path.exists()
    post_fea_change_log = json.loads(post_fea_change_log_path.read_text(encoding="utf-8"))
    print(f"  → change log keys : {sorted(post_fea_change_log.keys())}")
    assert post_fea_change_log["sample_id"] == sample_id
    assert post_fea_change_log["changed_features"]
    print("  ✓ State C post-FEA revision is traceable")
else:
    print("  ✓ blocked as expected until manual evidence is complete")


## What this notebook proves

- Real FEA feedback can drive a traceable CAD revision.
- The notebook blocks cleanly when manual evidence is incomplete.
- State B vs State C evidence is saved when the gate passes.
